# Exploring one Argoverse 2 scenario

Each scenario is 11 s of driving at 10 Hz (110 timesteps): 50 observed + 60 to predict.
One **focal agent** is the prediction target; everything else is context.

Things to notice as you run this:
- Coordinates are in a **city frame**, hundreds of meters from the origin — raw values are useless to a network until normalized.
- Most tracks are **partially observed** (NaN gaps) — masks, not imputation, handle this downstream.
- Lane centerlines basically sketch the road network — the map prior the model gets in stage 3.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from trajpred.scenario import load_scenario

RAW_VAL = Path(r"C:\data\av2\raw\val")
scenario_dir = sorted(RAW_VAL.iterdir())[3]
s = load_scenario(scenario_dir)
print(s.scenario_id)
print(f"{len(s.track_ids)} tracks, {len(s.lane_centerlines)} lanes")
print(f"focal agent: idx {s.focal_idx}, type {s.object_types[s.focal_idx]}")
obs_frac = 1 - np.isnan(s.positions[:, :, 0]).mean(axis=1)
print(f"observation fraction per track: min {obs_frac.min():.2f}, median {np.median(obs_frac):.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
for cl in s.lane_centerlines:
    ax.plot(cl[:, 0], cl[:, 1], color="0.8", lw=1, zorder=0)
for i in range(len(s.track_ids)):
    if i != s.focal_idx:
        ax.plot(s.positions[i, :, 0], s.positions[i, :, 1], color="tab:blue", alpha=0.4, lw=1)

# focal agent: history (steps 0-49) black, future (50-109) green
f = s.positions[s.focal_idx]
ax.plot(f[:50, 0], f[:50, 1], color="black", lw=2.5, label="focal history (5 s)")
ax.plot(f[50:, 0], f[50:, 1], color="tab:green", lw=2.5, label="focal future (6 s)")
ax.scatter(*f[49], color="black", s=60, zorder=5)
ax.set_aspect("equal"); ax.legend(); ax.set_title(s.scenario_id)
plt.show()

## Check your understanding

1. Change the scenario index and find one where the focal agent **turns**. Look at where the future diverges from the straight-line continuation of the history — that gap is exactly what separates a learned model from the constant-velocity baseline.
2. Print `s.positions[s.focal_idx, 49]` and `s.positions[s.focal_idx, 48]`. The difference × 10 Hz is the speed in m/s. Is this agent fast or slow?
3. Why do we split at index 49/50 and not 50/51? (Answer: 50 *observed* steps means indices 0..49; the model's \"present\" is index 49.)